# 03 — Data Cleaning

**Wind Turbine Predictive Maintenance & Failure Intelligence System**

## Objective
Produce a clean, leakage-safe version of each Wind Farm A dataset, ready for
feature engineering. Cleaning decisions are driven by what NB01/NB02 found, not
a generic checklist.

## What this notebook does
- Quantify missing values across all 22 datasets.
- Handle known data-quality issues (pitch-angle wrapping; unreliable min/max/std).
- Detect and handle frozen/stuck sensor readings.
- Decide how to treat metadata columns (`status_type_id`, `train_test`, `id`).
- Save cleaned per-dataset files to `data/processed/` (gitignored).

## Ground rules carried from NB01/NB02
- Use `_avg` signals; treat `_min`/`_max`/`_std` as unreliable (CARE docs).
- **`status_type_id` is a leakage risk** (entangled with the fault window) →
  drop it from the modelling feature set.
- No cross-time or cross-dataset statistic that could leak future info.
- Timestamps are anonymised; ordering is real, absolute dates are not.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid")

BASE = Path("..") / "data" / "raw" / "Wind Farm A"
DATASETS_DIR = BASE / "datasets"
PROCESSED_DIR = Path("..") / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

events = pd.read_csv(BASE / "event_info.csv", sep=";")
feat_desc = pd.read_csv(BASE / "feature_description.csv", sep=";")
csv_files = sorted(DATASETS_DIR.glob("*.csv"), key=lambda p: int(p.stem))

# Scan missing values across all 22 datasets
miss_rows = []
for p in csv_files:
    d = pd.read_csv(p, sep=";")
    na = d.isna().sum()
    na = na[na > 0]
    miss_rows.append({
        "event_id": int(p.stem),
        "rows": len(d),
        "cols_with_na": int((d.isna().sum() > 0).sum()),
        "total_na": int(d.isna().sum().sum()),
        "worst_col": na.idxmax() if len(na) else None,
        "worst_col_na": int(na.max()) if len(na) else 0,
    })

miss = pd.DataFrame(miss_rows)
print("Missing-value summary across 22 datasets:")
print(miss.to_string(index=False))
print(f"\nTotal NA across all datasets: {miss['total_na'].sum():,}")

Missing-value summary across 22 datasets:
 event_id  rows  cols_with_na  total_na     worst_col  worst_col_na
        0 54986             2         2 sensor_14_avg             1
        3 55487             2         4 sensor_14_avg             2
       10 53592             2         6 sensor_14_avg             3
       13 54010             0         0          None             0
       14 54197             0         0          None             0
       17 55090             2         6 sensor_14_avg             3
       22 53036             2         2 sensor_14_avg             1
       24 55003             0         0          None             0
       25 54712             0         0          None             0
       26 53702             2         2 sensor_14_avg             1
       38 54835             0         0          None             0
       40 56158             2         4 sensor_14_avg             2
       42 53886             2         6 sensor_14_avg             3
      

## 1. Missing values — trivial

40 NaNs across ~1.2M rows (mostly `sensor_14_avg`, 1–3 per file). Wind Farm A is
effectively complete, as the CARE docs state. These are isolated sensor dropouts,
not systematic gaps. We handle them with a **leakage-safe forward-fill within each
turbine's time series** (carry last valid reading forward — uses only past info),
falling back to column median only if a series starts with NaN.

In [2]:
# Column groups (established in NB01)
META_COLS = ["time_stamp", "asset_id", "id", "train_test", "status_type_id"]

# Angle sensors (from feature_description: is_angle == True)
angle_sensors = feat_desc.loc[feat_desc["is_angle"] == True, "sensor_name"].tolist()
print("Angle sensors (need wrap handling):", angle_sensors)

def clean_dataset(df):
    """Leakage-safe cleaning for one dataset (one turbine's time series)."""
    df = df.sort_values("id").reset_index(drop=True)  # ensure time order

    sensor_cols = [c for c in df.columns if c not in META_COLS]

    # 1. Missing values: forward-fill (past-only), then median fallback for leading NaNs
    df[sensor_cols] = df[sensor_cols].ffill()
    for c in sensor_cols:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].median())

    # 2. Pitch-angle wrapping (0 ≡ 360): encode angle sensors as sin/cos so the
    #    wrap-around discontinuity disappears. Applies to *_avg angle columns.
    for base in angle_sensors:
        col = f"{base}_avg"
        if col in df.columns:
            rad = np.deg2rad(df[col])
            df[f"{base}_sin"] = np.sin(rad)
            df[f"{base}_cos"] = np.cos(rad)

    return df

# Test on one file
test = pd.read_csv(DATASETS_DIR / "0.csv", sep=";")
cleaned = clean_dataset(test)
print("\nOriginal shape:", test.shape, "-> cleaned shape:", cleaned.shape)
print("NaNs after cleaning:", int(cleaned.isna().sum().sum()))
print("New angle features:", [c for c in cleaned.columns if c.endswith(("_sin","_cos"))])

Angle sensors (need wrap handling): ['sensor_1', 'sensor_2', 'sensor_5', 'sensor_42']

Original shape: (54986, 86) -> cleaned shape: (54986, 94)
NaNs after cleaning: 0
New angle features: ['sensor_1_sin', 'sensor_1_cos', 'sensor_2_sin', 'sensor_2_cos', 'sensor_5_sin', 'sensor_5_cos', 'sensor_42_sin', 'sensor_42_cos']


## 2. Frozen / stuck sensors

A frozen sensor repeats an identical value for long stretches — not missing, so it
survives NaN checks, but it's dead data. We check the key condition-monitoring
sensors for suspicious runs of identical consecutive values. Note: some flatness
is legitimate (a stopped turbine has near-zero, steady RPM), so we flag rather
than blindly remove, and interpret in context.

In [3]:
def max_run_length(series):
    """Longest run of identical consecutive values."""
    s = series.values
    if len(s) == 0:
        return 0
    # run-length via diff: where value changes, start a new run
    change = np.concatenate([[True], s[1:] != s[:-1]])
    run_ids = np.cumsum(change)
    return pd.Series(run_ids).value_counts().max()

# Check key condition sensors across all datasets
check_cols = ["sensor_11_avg", "sensor_12_avg", "sensor_13_avg", "sensor_14_avg",
              "sensor_38_avg", "sensor_41_avg", "power_30_avg", "sensor_18_avg"]

frozen_rows = []
for p in csv_files:
    d = pd.read_csv(p, sep=";", usecols=check_cols).sort_index()
    rec = {"event_id": int(p.stem)}
    for c in check_cols:
        rec[c] = max_run_length(d[c])
    frozen_rows.append(rec)

frozen = pd.DataFrame(frozen_rows).set_index("event_id")
print("Longest run of identical consecutive values, per sensor per dataset:")
print(frozen.to_string())
print("\nMax run length per sensor (across all datasets):")
print(frozen.max().sort_values(ascending=False).to_string())

Longest run of identical consecutive values, per sensor per dataset:
          sensor_11_avg  sensor_12_avg  sensor_13_avg  sensor_14_avg  sensor_38_avg  sensor_41_avg  power_30_avg  sensor_18_avg
event_id                                                                                                                       
0                    65            119             67             80             91             91            65             73
3                   167             76            124            587            149             91            51            141
10                  167             76            124            587            149             91            51            141
13                  171            228            125             88             76            111            72            118
14                   46             50             49             54             98            101            14             43
17                  167            

### Frozen-sensor findings

- Longest identical-value runs are modest: Worst is `sensor_14_avg` (gen bearing
  NDE) at 587 rows (~4 days), appearing in the same repeated turbine's files
  (asset 10). RPM (`sensor_18`, max 304) and power (max 108) flat runs are longer
  where expected — a **parked turbine** legitimately holds ~0 RPM/power for days.
- The `sensor_14` runs coincide with the files that had its few NaNs; forward-fill
  can slightly extend a flat run. Combined with cooling-to-ambient during idle,
  these are **plausibly legitimate parked periods, not broken sensors**.
- **Decision: do not remove.** Dropping flat runs would delete real idle-operation
  data and risk bias. We flag `sensor_14`'s 587-run as a minor known limitation and
  retain all rows. Operating state will instead be captured as a *feature* in NB04
  (e.g. power/RPM-based running indicator)

## 3. Apply cleaning and save processed datasets

Run the leakage-safe cleaning on all 22 datasets and save to `data/processed/`
(gitignored). Each cleaned file keeps its metadata columns and gains sin/cos angle
features. These processed files are the input to feature engineering (NB04).

In [4]:
manifest = []
for p in csv_files:
    d = pd.read_csv(p, sep=";")
    cleaned = clean_dataset(d)
    out_path = PROCESSED_DIR / f"{p.stem}_clean.csv"
    cleaned.to_csv(out_path, index=False)
    manifest.append({
        "event_id": int(p.stem),
        "rows": len(cleaned),
        "cols": cleaned.shape[1],
        "na_after": int(cleaned.isna().sum().sum()),
        "saved": out_path.name,
    })

manifest_df = pd.DataFrame(manifest)
print("Cleaned & saved all datasets:")
print(manifest_df.to_string(index=False))
print(f"\nTotal NaNs across all cleaned files: {manifest_df['na_after'].sum()}")
print(f"All files -> {PROCESSED_DIR}")

Cleaned & saved all datasets:
 event_id  rows  cols  na_after        saved
        0 54986    94         0  0_clean.csv
        3 55487    94         0  3_clean.csv
       10 53592    94         0 10_clean.csv
       13 54010    94         0 13_clean.csv
       14 54197    94         0 14_clean.csv
       17 55090    94         0 17_clean.csv
       22 53036    94         0 22_clean.csv
       24 55003    94         0 24_clean.csv
       25 54712    94         0 25_clean.csv
       26 53702    94         0 26_clean.csv
       38 54835    94         0 38_clean.csv
       40 56158    94         0 40_clean.csv
       42 53886    94         0 42_clean.csv
       45 53739    94         0 45_clean.csv
       51 54436    94         0 51_clean.csv
       68 54358    94         0 68_clean.csv
       69 54813    94         0 69_clean.csv
       71 54744    94         0 71_clean.csv
       72 54082    94         0 72_clean.csv
       73 54042    94         0 73_clean.csv
       84 53772    94    